# Moteur de recommandation sportive - MSPR 2

Predire le type d'entrainement (`workout_type`) a partir du profil utilisateur.
Modele : RandomForest. Tout le raisonnement est ici, du chargement des donnees a l'evaluation.


## 1. Chargement + exploration des donnees


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay

# les CSV sont a la racine du projet (exports Supabase _rows.csv)
users = pd.read_csv('../users_rows.csv')
sessions = pd.read_csv('../workout_sessions_rows.csv')
print('users   :', users.shape)
print('sessions:', sessions.shape)
sessions.head()


In [ ]:
# repartition de la cible
sessions['workout_type'].value_counts()


## 2. Preparation des features


In [ ]:
FEATURES = ['age', 'gender', 'bmi', 'body_fat_pct', 'goal', 'duration_min', 'calories_burned']
TARGET = 'workout_type'

# join sessions + profil user
df = sessions.merge(users, left_on='user_id', right_on='id', suffixes=('_session', '_user'))
df = df[FEATURES + [TARGET]].dropna()

X = df[FEATURES].copy()
y = df[TARGET].copy()

# encodage gender + goal
encoders = {}
for col in ['gender', 'goal']:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('train:', X_train.shape, '| test:', X_test.shape)
X.head()


## 3. Entrainement (RandomForest)


In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

# cross-validation pour valider la stabilite
scores = cross_val_score(rf, X, y, cv=5)
print('cv accuracy = %.3f (+/- %.3f)' % (scores.mean(), scores.std()))


## 4. Metriques + matrice de confusion


In [ ]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf.classes_)
disp.plot(cmap='Blues')
plt.title('Matrice de confusion')
plt.show()


In [ ]:
# importance des features
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind='barh', title="Importance des features")
plt.tight_layout()
plt.show()


## 5. Comparaison avant/apres ajustement (n_estimators)


In [ ]:
resultats = {}
for n in [100, 200]:
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X_train, y_train)
    resultats[n] = accuracy_score(y_test, m.predict(X_test))
    print('n_estimators=%d -> accuracy test = %.3f' % (n, resultats[n]))

plt.bar([str(k) for k in resultats], list(resultats.values()), color=['#9ecae1', '#3182bd'])
plt.ylim(0, 1)
plt.ylabel('accuracy test')
plt.title('Avant (100) vs apres (200) ajustement')
plt.show()
